In [ ]:
DATA PROCESSING, CLEANING, AND MANIPULATION

In [2]:
import pandas as pd

inspection = pd.read_csv("G:\\My Drive\\Data Analysis Projects\\Restaurant inspection\\Data\\DOHMH_New_York_City_Restaurant_Inspection_Results.csv")

print(inspection.head())

      CAMIS                          DBA       BORO BUILDING       STREET  \
0  50104876            NOODLE SUPER NO I  Manhattan      265     1 AVENUE   
1  50174387              TREATS OF KOREA     Queens     3150  STEINWAY ST   
2  50122756  MELLER'S SPORTS HUB & GRILL  Manhattan     1702     2 AVENUE   
3  41408131             CHRIS RESTAURANT   Brooklyn     1866    86 STREET   
4  50138007               PEARL OF CHINA   Brooklyn     8411     3 AVENUE   

   ZIPCODE       PHONE CUISINE DESCRIPTION INSPECTION DATE  \
0  10003.0  2125290539             Chinese      08/24/2022   
1  11103.0  8453232965                 NaN      01/01/1900   
2  10128.0  9175964244            American      08/04/2025   
3  11214.0  3474623755              Polish      05/20/2024   
4  11209.0  7188334281             Chinese      08/16/2023   

                                            ACTION  ...  \
0  Violations were cited in the following area(s).  ...   
1                                             

In [3]:
inspection.columns = (inspection.columns
              .str.strip()
              .str.replace(' ', '_')
              .str.replace('-', '_')
              .str.replace('/', '_')
              .str.lower())


print("New Standardized Columns:")
print(inspection.columns.tolist())

New Standardized Columns:
['camis', 'dba', 'boro', 'building', 'street', 'zipcode', 'phone', 'cuisine_description', 'inspection_date', 'action', 'violation_code', 'violation_description', 'critical_flag', 'score', 'grade', 'grade_date', 'record_date', 'inspection_type', 'latitude', 'longitude', 'community_board', 'council_district', 'census_tract', 'bin', 'bbl', 'nta', 'location_point1']


In [4]:
inspection.drop(['building', 'phone', 'community_board', 'council_district', 'census_tract', 'bin', 'bbl', 'nta', 'location_point1' ], axis=1, inplace=True, errors='ignore')

In [5]:
inspection.isna().sum()

camis                         0
dba                           6
boro                          0
street                        3
zipcode                    2911
cuisine_description        3699
inspection_date               0
action                     3699
violation_code             5851
violation_description      5851
critical_flag                 0
score                     15926
grade                    148044
grade_date               155868
record_date                   0
inspection_type            3699
latitude                    413
longitude                   413
dtype: int64

In [6]:
inspection = inspection[inspection['critical_flag'] != 'Not Applicable']

print(inspection['critical_flag'].unique())

['Critical' 'Not Critical']


In [35]:
# 1. Address 'ZIPCODE' with "Unknown"
# Ensures Zipcode is a string first to prevent losing leading zeros
inspection['zipcode'] = inspection['zipcode'].astype(str)

#  Cleaning the string (remove .0 if it was imported as a float)
inspection['zipcode'] = inspection['zipcode'].str.replace(r'\.0$', '', regex=True)

# 3. Filling missing/null/nan with "Unknown"
inspection['zipcode'] = inspection['zipcode'].replace(['nan', 'None', 'nan', ''], 'Unknown')

# 4. Standardizing length (Optional: helps catch data entry errors)
# If it's not 5 digits and not "Unknown", it might be a 'Bad Entry'
inspection['zipcode'] = inspection['zipcode'].apply(lambda x: x if len(x) == 5 or x == "Unknown" else "Invalid")

In [36]:
# 2. Address 'SCORE' with the mean
inspection['score'] = inspection['score'].fillna(inspection['score'].mean())

In [77]:
# 3. Address 'GRADE' with "NA"
inspection['grade'] = inspection['grade'].fillna("NA")

In [78]:
# 4. Address 'GRADE DATE' with "Unknown"
inspection['grade_date'] = inspection['grade_date'].fillna("Unknown")

In [11]:
# 5. Address Geospatial data with 0.0 (numeric)
inspection['latitude'] = inspection['latitude'].fillna(0.0)
inspection['longitude'] = inspection['longitude'].fillna(0.0)

In [79]:
# 6. Final validation
if inspection.isnull().sum().sum() == 0:
    print("✅ Data is clean! No missing values detected.")
else:
    print("⚠️ Warning: Missing values still exist in the following columns:")
    print(inspection.isnull().sum()[inspection.isnull().sum() > 0])

✅ Data is clean! No missing values detected.


In [ ]:
# 4. Standardising cuisine categories using a single identifier

In [74]:
# 1. Defining the mapping of variations to the single identifier
cuisine_map = {
    'Creole': 'Creole/Cajun',
    'Cajun': 'Creole/Cajun',
    'Hotdogs': 'Hotdogs/Pretzels',
    'Salads': 'Soup/Salads/Sandwiches/Mixed Buffet',
    'Sandwiches': 'Soup/Salads/Sandwiches/Mixed Buffet',
    'Soups': 'Soup/Salads/Sandwiches/Mixed Buffet',
    'Sandwiches/Salads/Mixed Buffet': 'Soup/Salads/Sandwiches/Mixed Buffet',
    'Soups/Salads/Sandwiches': 'Soup/Salads/Sandwiches/Mixed Buffet'
}

# 2. Ensuring the column is a string/object type first
inspection['cuisine_description'] = inspection['cuisine_description'].astype(str)

# 3. Applying the mapping to the cuisine column
inspection['cuisine_description'] = inspection['cuisine_description'].replace(cuisine_map)

In [64]:
inspection.info()

<class 'pandas.core.frame.DataFrame'>
Index: 281295 entries, 0 to 288887
Data columns (total 18 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   camis                  281295 non-null  int64         
 1   dba                    281295 non-null  object        
 2   boro                   281295 non-null  category      
 3   street                 281295 non-null  object        
 4   zipcode                281295 non-null  object        
 5   cuisine_description    281295 non-null  object        
 6   inspection_date        281295 non-null  datetime64[ns]
 7   action                 281295 non-null  category      
 8   violation_code         281295 non-null  object        
 9   violation_description  281295 non-null  object        
 10  critical_flag          281295 non-null  category      
 11  score                  281295 non-null  float64       
 12  grade                  281295 non-null  category 

In [65]:
# 1. Converting Date columns
date_cols = ['inspection_date', 'grade_date', 'record_date']
for col in date_cols:
    # errors='coerce' will handle any remaining "Unknown" or malformed strings by turning them into NaT (Not a Time)
    inspection[col] = pd.to_datetime(inspection[col], format='%m/%d/%Y', errors='coerce')

In [66]:
# 2. Converting to Categorical to save memory and improve speed
cat_cols = ['boro', 'critical_flag', 'grade', 'action', 'cuisine_description']
for col in cat_cols:
    inspection[col] = inspection[col].astype('category')

In [67]:
# 3. Ensuring identifiers don't act like floats
inspection['zipcode'] = inspection['zipcode'].astype(str).str.replace('.0', '', regex=False)


print(inspection.dtypes)

camis                             int64
dba                              object
boro                           category
street                           object
zipcode                          object
cuisine_description            category
inspection_date          datetime64[ns]
action                         category
violation_code                   object
violation_description            object
critical_flag                  category
score                           float64
grade                          category
grade_date               datetime64[ns]
record_date              datetime64[ns]
inspection_type                  object
latitude                        float64
longitude                       float64
dtype: object


In [68]:
duplicate_rows = inspection.duplicated()
print(f"Number of duplicate rows: {duplicate_rows.sum()}")

duplicates = inspection[inspection.duplicated()]
print(duplicates)

Number of duplicate rows: 6
           camis                      dba    boro         street  zipcode  \
89493   50001285  Y B ENTERTAINMENT MANOR  Queens  PRINCE STRRET  INVALID   
124451  50001285  Y B ENTERTAINMENT MANOR  Queens  PRINCE STRRET  INVALID   
182980  50001285  Y B ENTERTAINMENT MANOR  Queens  PRINCE STRRET  INVALID   
207536  50001285  Y B ENTERTAINMENT MANOR  Queens  PRINCE STRRET  INVALID   
259773  50001285  Y B ENTERTAINMENT MANOR  Queens  PRINCE STRRET  INVALID   
268607  50001285  Y B ENTERTAINMENT MANOR  Queens  PRINCE STRRET  INVALID   

       cuisine_description inspection_date  \
89493               Korean      2019-06-28   
124451              Korean      2019-06-28   
182980              Korean      2019-06-28   
207536              Korean      2019-06-28   
259773              Korean      2019-06-28   
268607              Korean      2019-06-28   

                                                 action violation_code  \
89493   Violations were cited in th

In [ ]:
# Six duplicate rows were found. However, additional analysis of the dataset revealed that these rows were not duplicates because each row had a unique identifier (CAMIS) and a distinct violation description. As a result, no further cleaning was done.

In [72]:
import re

# Identify object columns
obj_cols = inspection.select_dtypes(include=['object']).columns

for col in obj_cols:
    # 1. Standardizing Casing 
    inspection[col] = inspection[col].str.upper()
    
    # 2. Trimming Whitespaces
    inspection[col] = inspection[col].str.strip()
    
    # 3. Removing Special Characters (keeping only letters, numbers, and spaces)
    # Regex removes symbols like $, #, and extra quotes found in the DBA column
    inspection[col] = inspection[col].str.replace(r'[^A-Z0-9 ]', '', regex=True)
    
    # 4. Handling Double Spaces (often left behind after stripping symbols)
    inspection[col] = inspection[col].str.replace(r'\s+', ' ', regex=True)

In [75]:
import pandas as pd
import sqlite3


# 1. Establishing a connection
# This creates the file 'restaurant_inspections.db' in the working directory
with sqlite3.connect('restaurant_inspections.db') as conn:

# 2. Writing the data to a table
# 'name' is the table name for the db
# 'if_exists' replaces the table if it already exists
    inspection.to_sql(name='inspections', con=conn, if_exists='replace', index=False)

# 3. To make sure the database is correctly built before downloading it
import os
file_stats = os.stat('restaurant_inspections.db')
print(f"Database size: {file_stats.st_size / (1024 * 1024):.2f} MB")

Database size: 116.44 MB
